# sklearn Pipeline — Rock vs. Mine Classification
## Neural networks for energy applications — Day 1, Exercise 1

In this exercise we build a complete machine-learning pipeline using scikit-learn to classify sonar signals as either **Rock (R)** or **Mine/Metal (M)**.

The dataset (`sonar.all-data.csv`) contains 208 samples, each with 60 continuous features representing sonar signal energy at different frequency bands. The task is a binary classification problem.

By the end of this notebook you will:
- Load and explore a real-world sonar dataset
- Encode the labels with an explicit, printed mapping (Mine = 1, Rock = 0)
- Build sklearn `Pipeline` objects that combine preprocessing and modelling
- Compare Logistic Regression, Random Forest, and SVM classifiers with cross-validation on the training rows
- Tune the chosen model with grid search, then evaluate it **once** on the held-out test rows
- Visualize results with bar charts, box plots, and a confusion matrix

**Evaluation protocol.** The 42 test rows are set aside in section 5 and used only once, in section 12. Every choice before that (model family, hyperparameters) is made with 5-fold cross-validation on the 166 training rows.

---
### 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

np.random.seed(42)
print("Imports complete.")

---
### 2 — Load the Sonar Dataset

The CSV has no header row. There are 61 columns: the first 60 are continuous sonar features and the last column is the class label (`R` = Rock, `M` = Mine/Metal).

In [ ]:
df = pd.read_csv("sonar.all-data.csv", header=None)

# Assign readable column names
feature_cols = [f"feature_{i}" for i in range(60)]
df.columns = feature_cols + ["label"]

print(f"Shape: {df.shape}")
df.head()

---
### 3 — Explore the Data

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes.value_counts())
print("\nMissing values:", df.isnull().sum().sum())

In [ ]:
df.describe().T.head(10)

In [ ]:
# Class distribution (Rock first: Rock blue, Mine orange, as in the course slides)
counts = df["label"].value_counts().reindex(["R", "M"])
print("Class distribution:")
print(counts)

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind="bar", color=["steelblue", "coral"], edgecolor="k", ax=ax)
ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_title("Class Distribution (R=Rock, M=Mine)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

---
### 4 — Prepare the Data

Separate features from labels and encode the string labels with an **explicit mapping**: Mine (`M`) = 1, the positive class, and Rock (`R`) = 0. Every Sonar lab in this course uses this mapping (Lab 2 this afternoon and the PyTorch lab on Day 2). `LabelEncoder` would sort the labels alphabetically and give M = 0, R = 1, silently swapping the positive class, so we write the mapping out and print it: precision and recall depend on which class is positive.

In [ ]:
X = df[feature_cols].values
y_raw = df["label"].values

# Explicit label mapping: Mine = 1 (positive class), Rock = 0 -- the same in every Sonar lab
LABEL_MAP = {"R": 0, "M": 1}
CLASS_NAMES = ["Rock", "Mine"]          # position = encoded label (0, 1)
y = df["label"].map(LABEL_MAP).to_numpy()

print(f"Features shape: {X.shape}")
print(f"Labels shape:   {y.shape}")
print(f"Label mapping:  {LABEL_MAP}  (1 = Mine, the positive class)")
print(f"Class counts:   Rock = {np.sum(y == 0)}, Mine = {np.sum(y == 1)}")

---
### 5 — Train / Test Split and Cross-Validation Folds

We use an 80/20 split with stratification to preserve class balance. We split the row numbers and then index the arrays (the same partition as splitting `X` and `y` directly), so that section 10 can put the training rows back into file order. The split is stratified on the `M`/`R` label column; Lab 2 uses the identical call, so both labs hold out the same 42 test echoes.

The **42 test rows are then set aside until section 12**. To compare and tune models we use one set of 5 shuffled, stratified folds on the 166 training rows, shared by every model, so that differences between scores come from the models and not from the folds.

In [ ]:
# Split row numbers, then index the arrays (same partition as splitting X and y directly)
idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=42, stratify=y_raw
)
X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

# One set of shuffled, stratified folds on the training rows, shared by every model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples (untouched until section 12)")
print(f"Train class counts [Rock, Mine]: {np.bincount(y_train)}")
print(f"Test class counts [Rock, Mine]:  {np.bincount(y_test)}")

---
### 6 — Pipeline 1: StandardScaler + Logistic Regression

An sklearn `Pipeline` chains a sequence of transforms followed by a final estimator. Inside `cross_val_score` the whole pipeline is refitted on the training part of each fold, so the scaler never sees the validation fold (no data leakage).

We estimate each pipeline's accuracy with 5-fold cross-validation on the training rows, using the shared `cv` folds. The test rows are not used here.

In [ ]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring="accuracy")
print(f"Logistic Regression CV accuracy: {scores_lr.mean():.3f} +/- {scores_lr.std():.3f}")
print(f"Fold scores: {scores_lr.round(3)}")

---
### 7 — Pipeline 2: StandardScaler + Random Forest

In [ ]:
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42)),
])

scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring="accuracy")
print(f"Random Forest CV accuracy: {scores_rf.mean():.3f} +/- {scores_rf.std():.3f}")
print(f"Fold scores: {scores_rf.round(3)}")

---
### 8 — Pipeline 3: StandardScaler + SVC

In [ ]:
pipe_svc = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", random_state=42)),
])

scores_svc = cross_val_score(pipe_svc, X_train, y_train, cv=cv, scoring="accuracy")
print(f"SVC CV accuracy: {scores_svc.mean():.3f} +/- {scores_svc.std():.3f}")
print(f"Fold scores: {scores_svc.round(3)}")

---
### 9 — Compare All Models and Choose One

Compare the cross-validated accuracy of the three pipelines on the same folds, next to a majority-class baseline, and choose the model family to tune: the one with the highest mean CV accuracy.

**Why cross-validation and not the test set?** Choosing a model by its test accuracy spends the test set: once a score has influenced a decision, it no longer estimates performance on new data. With about 33 echoes per fold, one echo moves a fold score by 3 points, so compare the gaps between models with the fold-to-fold spread before calling one of them "best".

In [ ]:
model_names = ["Logistic Regression", "Random Forest", "SVC"]
pipelines = [pipe_lr, pipe_rf, pipe_svc]
cv_scores = [scores_lr, scores_rf, scores_svc]
cv_means = [s.mean() for s in cv_scores]

# Baseline on the same folds: always predict the majority class of the training part
majority = cross_val_score(DummyClassifier(strategy="most_frequent"),
                           X_train, y_train, cv=cv, scoring="accuracy")

print(f"{'Model':20s} {'CV mean':>8s} {'CV sd':>7s}")
for name, s in zip(model_names, cv_scores):
    print(f"{name:20s} {s.mean():8.3f} {s.std():7.3f}")
print(f"{'Majority class':20s} {majority.mean():8.3f} {majority.std():7.3f}")

# Bar chart: fold means with the fold-to-fold standard deviation
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["steelblue", "seagreen", "coral"]
bars = ax.bar(model_names, cv_means, yerr=[s.std() for s in cv_scores], capsize=6,
              color=colors, edgecolor="k")
ax.axhline(majority.mean(), color="grey", linestyle="--", label="Majority-class baseline")
ax.set_ylabel("5-fold CV accuracy (training rows)")
ax.set_title("Model Comparison — Sonar Classification")
ax.set_ylim(0, 1.05)
for bar, m in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width() / 2, 0.05, f"{m:.3f}",
            ha="center", color="white", fontsize=12, fontweight="bold")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# Choose the model family with the highest mean CV accuracy
best_idx = int(np.argmax(cv_means))
best_name = model_names[best_idx]
best_pipe = pipelines[best_idx]
print(f"\nChosen by cross-validation: {best_name} (mean CV accuracy = {cv_means[best_idx]:.3f})")

**Reading the comparison.** In the reference run, random forest and SVM are level (their means differ by less than 0.001, far less than the fold-to-fold spread) and both clearly beat logistic regression and the majority-class baseline. The rule "highest mean CV accuracy" picks one of them; either would be a defensible choice. What matters is that the rule was fixed before looking, and that the test rows played no part.

---
### 10 — Cross-Validation Comparison

A single train/test split can be misleading, and so can a single summary number. First we look at the fold-level scores behind the means from sections 6–8.

Then a **row-order check**. The file is not in random order: all rocks come first, then all mines, and within each class neighbouring rows may be related (the UCI documentation says the original returns were numbered in increasing aspect angle). We rerun the same pipelines on the training rows put back into **file order**, with 5 stratified folds **without shuffling**, so that each validation fold is a contiguous block of the file. This is a diagnostic only: it uses the training rows alone and does not change the choice made in section 9.

In [ ]:
cv_results = dict(zip(model_names, cv_scores))

# Box plot of the fold scores (shuffled folds, training rows)
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(list(cv_results.values()), tick_labels=list(cv_results.keys()), patch_artist=True,
           boxprops=dict(facecolor="lightskyblue", edgecolor="k"),
           medianprops=dict(color="red", linewidth=2))
ax.set_ylabel("Accuracy")
ax.set_title("5-Fold Cross-Validation Comparison (shuffled folds, training rows)")
ax.set_ylim(0.5, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Row-order check (diagnostic only): training rows back in FILE order, unshuffled stratified folds
file_order = np.argsort(idx_train)
X_train_file, y_train_file = X_train[file_order], y_train[file_order]
cv_file = StratifiedKFold(n_splits=5, shuffle=False)

print(f"{'Model':20s} {'shuffled folds':>15s} {'file-order folds':>17s}")
for name, pipe, s in zip(model_names, pipelines, cv_scores):
    s_file = cross_val_score(pipe, X_train_file, y_train_file, cv=cv_file, scoring="accuracy")
    print(f"{name:20s} {s.mean():15.3f} {s_file.mean():17.3f}   file-order folds: {s_file.round(2)}")

**Reading the check.** When each validation fold is a contiguous block of the file, every pipeline loses 10 to 25 points (reference run: about 0.59 to 0.65, against 0.75 to 0.83 with shuffled folds). Neighbouring rows in the file are more alike than random pairs, so a random split may be optimistic about echoes recorded under new conditions. In energy data the equivalent structure is usually known (wells, pads, pumps, time periods): use `GroupKFold` or time-ordered splits whenever that information exists.

---
### 11 — GridSearchCV on the Chosen Model

We tune the model family chosen in section 9. `GridSearchCV` runs cross-validation on the training rows for every setting in the grid, using the same shared folds. Because pipeline parameter names are prefixed with the step name (e.g. `clf__C`), we can tune the estimator inside the pipeline, and the scaler is refitted within every fold.

The best CV score is itself slightly optimistic: it is the maximum over all the settings evaluated on the same folds. The test rows are still untouched.

In [ ]:
# Parameter grids for each model family (pipeline step name + "__" + parameter name)
param_grids = {
    "Logistic Regression": {
        "clf__C": [0.01, 0.1, 1, 10, 100],
    },
    "Random Forest": {
        "clf__n_estimators": [50, 100, 200],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_split": [2, 5],
    },
    "SVC": {
        "clf__C": [0.1, 1, 10, 100],
        "clf__gamma": ["scale", "auto", 0.01, 0.001],
        "clf__kernel": ["rbf"],
    },
}

print(f"Running GridSearchCV on: {best_name}")
print(f"Parameter grid: {param_grids[best_name]}\n")

grid_search = GridSearchCV(
    best_pipe,
    param_grids[best_name],
    cv=cv,                  # the same shuffled folds as in sections 6-10
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)   # training rows only

print(f"Best CV accuracy: {grid_search.best_score_:.3f}")
print(f"Best parameters:  {grid_search.best_params_}")

---
### 12 — Final Test Evaluation and Confusion Matrix

The choice is now locked: the model family (section 9) and its hyperparameters (section 11) were chosen with cross-validation on the training rows, and `GridSearchCV` has refitted the best setting on all 166 training rows. We evaluate it **once** on the 42 held-out test rows, next to the majority-class baseline, and plot its confusion matrix. Do not go back and change the model because of this score: that would turn the test set into a validation set.

Each test echo is worth 2.4 percentage points (1/42), so report the score together with its count.

In [ ]:
# The only use of the test set: the locked model, evaluated once
final_model = grid_search.best_estimator_
y_pred_test = final_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
n_correct = int((y_pred_test == y_test).sum())

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_acc = baseline.score(X_test, y_test)

print(f"Final model: {best_name} with {grid_search.best_params_}")
print(f"Test accuracy: {test_acc:.3f} ({n_correct} of {len(y_test)} echoes)")
print(f"Majority-class baseline on the same rows: {baseline_acc:.3f}")
print("\nClassification report (test set; positive class = Mine):")
print(classification_report(y_test, y_pred_test, target_names=CLASS_NAMES))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Confusion Matrix (counts) — {best_name}")

# Normalized
cm_norm = confusion_matrix(y_test, y_pred_test, normalize="true")
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASS_NAMES)
disp_norm.plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title(f"Confusion Matrix (normalized) — {best_name}")

plt.tight_layout()
plt.show()

---
### Summary

In this exercise we:

1. Loaded and explored the UCI Sonar dataset (208 samples, 60 features).
2. Encoded the labels with an explicit, printed mapping (Mine = 1, Rock = 0) and split the data 80/20 with stratification.
3. Built three sklearn `Pipeline` objects, each combining `StandardScaler` with a different classifier.
4. Compared Logistic Regression, Random Forest, and SVC with 5-fold cross-validation on the training rows, next to a majority-class baseline, and chose one family.
5. Checked that unshuffled folds in file order give much lower scores: row order carries structure.
6. Applied `GridSearchCV` on the same folds to tune the chosen model's hyperparameters.
7. Evaluated the locked model once on the 42 test rows and visualized its confusion matrix.

**Key takeaway:** sklearn `Pipeline` objects are the recommended way to chain preprocessing and modelling steps. They prevent data leakage during cross-validation and grid search, and make the entire workflow reproducible with a single `.fit()` / `.predict()` call. Choose with cross-validation on the training rows, then touch the test set once.